# Vòng train hỏng mà không báo lỗi

Một đồng nghiệp viết MLP numpy để phân loại 3 nhánh xoắn ốc. Code chạy hết, không exception, nhưng
accuracy chỉ quanh 33% (đoán mò) và gradient check không qua. Trong notebook có **4 bug "im lặng"**,
mỗi bug một chỗ; bạn tìm, sửa, và ghi lại trong `BUGS.md`.

**Cách làm bài**
- Sửa ngay trong các cell code của notebook. Không đổi dữ liệu (`make_spirals`) và siêu tham số mặc định của
  `train`: bug phải sửa ở gốc, không bù bằng lr hay số epoch.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) để chạy thử; test bỏ qua chúng. Thêm bao nhiêu cell
  `explore` tuỳ bạn để in, vẽ, đo.
- Chấm: `uv run pytest tracks/ai-engineer/modules/01-nn-from-scratch/exercises/advanced/02-debug-broken-training -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import time

import numpy as np

## 1. Dữ liệu và model

3 nhánh xoắn ốc trong mặt phẳng, 200 điểm mỗi nhánh; model `MLP(2, 64, 3)`, softmax cross-entropy.
Quy ước như phần lý thuyết: batch theo dòng, `W` shape `(in, out)`, $$\text{out} = xW + b$$.

In [ ]:
def make_spirals(n_per_class: int, n_classes: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """`n_classes` nhánh xoắn ốc trong mặt phẳng, mỗi nhánh `n_per_class` điểm. X float64 (N, 2), y int64."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.05, 1.0, n_per_class)
        t = k * 2 * np.pi / n_classes + 8 * r + rng.normal(0, 0.2, n_per_class)
        X.append(np.c_[r * np.sin(t), r * np.cos(t)])
        y.append(np.full(n_per_class, k))
    return np.concatenate(X), np.concatenate(y).astype(np.int64)

In [ ]:
class MLP:
    """Linear → ReLU → Linear → ReLU → Linear. Hai tầng ẩn cùng cỡ `hidden`, tầng cuối ra logits.

    W khởi tạo He, b bằng 0, float64. Gradient được cộng dồn vào `self.grads` (như `.grad` của PyTorch).
    """

    NAMES = ["W1", "b1", "W2", "b2", "W3", "b3"]

    def __init__(self, n_in: int, hidden: int, n_out: int, seed: int):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, np.sqrt(2 / n_in), (n_in, hidden))
        self.W2 = rng.normal(0, np.sqrt(2 / hidden), (hidden, hidden))
        self.W3 = rng.normal(0, np.sqrt(2 / hidden), (hidden, n_out))
        self.b1 = self.b2 = np.zeros(hidden)
        self.b3 = np.zeros(n_out)
        self.grads = {name: np.zeros_like(getattr(self, name)) for name in self.NAMES}

    def zero_grad(self) -> None:
        for g in self.grads.values():
            g[...] = 0

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        self.z1 = X @ self.W1 + self.b1
        self.h1 = np.maximum(self.z1, 0)
        self.z2 = self.h1 @ self.W2 + self.b2
        self.h2 = np.maximum(self.z2, 0)
        return self.h2 @ self.W3 + self.b3

    def loss_and_backward(self, X: np.ndarray, y: np.ndarray) -> float:
        """Softmax cross-entropy trung bình trên batch; backward, cộng gradient vào `self.grads`."""
        logits = self.forward(X)
        n = len(X)
        z = logits - logits.max(axis=1, keepdims=True)
        p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
        loss = float(-np.log(p[np.arange(n), y]).mean())

        d = p.copy()
        d[np.arange(n), y] -= 1
        g = {"W3": self.h2.T @ d, "b3": d.sum(axis=0)}
        dz2 = (d @ self.W3.T) * (self.z2 > 0)
        g["W2"], g["b2"] = self.h1.T @ dz2, dz2.sum(axis=0)
        dz1 = (dz2 @ self.W2.T) * (self.z1 > 0)
        g["W1"], g["b1"] = self.X.T @ dz1, dz1.sum(axis=0)
        for name, value in g.items():
            self.grads[name] += value
        return loss

    def params(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """[(W1, dW1), (b1, db1), ...]: chính các mảng của model."""
        return [(getattr(self, name), self.grads[name]) for name in self.NAMES]

## 2. Vòng train

SGD momentum kiểu PyTorch ($$v \leftarrow \mu v + g,\ \theta \leftarrow \theta - \eta v$$), batch 32, 60 epoch.
Bản đúng đạt khoảng 99% test accuracy và chạy dưới 1 giây.

In [ ]:
def shuffle(X: np.ndarray, y: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Xáo thứ tự mẫu cho một epoch."""
    return X[rng.permutation(len(X))], y[rng.permutation(len(y))]


def accuracy(model: MLP, X: np.ndarray, y: np.ndarray) -> float:
    return float((model.forward(X).argmax(axis=1) == y).mean())


def train(seed: int, epochs: int = 60, lr: float = 0.05, batch_size: int = 32, momentum: float = 0.9) -> float:
    """Train MLP(2, 64, 3) trên 600 điểm xoắn ốc bằng SGD momentum; trả accuracy trên 600 điểm test."""
    X_train, y_train = make_spirals(200, 3, seed)
    X_test, y_test = make_spirals(200, 3, seed + 1000)
    model = MLP(2, 64, 3, seed)
    rng = np.random.default_rng(seed)
    velocity = [np.zeros_like(p) for p, _ in model.params()]
    for epoch in range(epochs):
        Xs, ys = shuffle(X_train, y_train, rng)
        for start in range(0, len(Xs), batch_size):
            model.loss_and_backward(Xs[start:start + batch_size], ys[start:start + batch_size])
            for (p, g), v in zip(model.params(), velocity):
                v *= momentum
                v += g
                p -= lr * v
    return accuracy(model, X_test, y_test)

## 3. Gradient check

So gradient giải tích với sai phân trung tâm trên một MLP nhỏ, mọi phần tử của mọi tham số.

In [ ]:
def max_gradient_error(eps: float = 1e-5) -> float:
    """Sai lệch tương đối lớn nhất giữa gradient giải tích và sai phân trung tâm, mọi phần tử mọi tham số."""
    rng = np.random.default_rng(0)
    model = MLP(2, 5, 3, seed=1)
    for name in ["b1", "b2", "b3"]:
        b = getattr(model, name)
        b[...] = rng.normal(0, 0.1, b.shape)          # bias khác 0 để kiểm cả db
    X, y = rng.normal(size=(6, 2)), rng.integers(0, 3, 6)
    model.zero_grad()
    model.loss_and_backward(X, y)
    analytic = [g.copy() for _, g in model.params()]
    worst = 0.0
    for (p, _), ga in zip(model.params(), analytic):
        for i in np.ndindex(p.shape):
            old = p[i]
            p[i] = old + eps
            plus = model.loss_and_backward(X, y)
            p[i] = old - eps
            minus = model.loss_and_backward(X, y)
            p[i] = old
            numeric = (plus - minus) / (2 * eps)
            worst = max(worst, abs(numeric - ga[i]) / max(abs(numeric) + abs(ga[i]), 1e-12))
    return worst


def gradient_check_passes() -> bool:
    return bool(max_gradient_error() < 1e-6)

In [ ]:
# explore: chạy thử
for seed in [0, 1, 2]:
    start = time.perf_counter()
    print(f"seed {seed}: test accuracy {train(seed):.4f} ({time.perf_counter() - start:.1f} s)")
print(f"gradient check: sai lệch lớn nhất {max_gradient_error():.2e}, qua: {gradient_check_passes()}")

In [ ]:
# explore: chỗ thử của bạn (vd. in loss từng epoch, xem vài cặp (x, y) sau khi xáo, đo norm gradient,
# gradient check từng tham số, overfit một batch nhỏ)

## 4. Ghi lại

Khi `train(seed)` đạt ≥ 0.95 với seed 0, 1, 2 và `gradient_check_passes()` là `True`, điền `BUGS.md`: mỗi bug
một mục (triệu chứng → cách phát hiện → sửa). Rồi làm phần **trắc nghiệm** của bài trong app.